# One cascade run on the random grids

**Question:** does the datacenter *placement effect* (where you connect it changes how much demand the grid can take
and how much fails) also appear in the degree-preserving random grids, or is it a property of the real grid?

**Setup:** Sergiu's `src/cascade.py` unchanged (`add_datacenter`, `cascade`, `demand_threshold`), **one α = 0.2, one demand = 60**,
the same settings as his `03_cascade.ipynb` (demand 60 is where the real grid shows the biggest placement effect).

**Locations:** the same rule as Sergiu's candidate list: the 20 most loaded substations at ≥ 380 kV, **picked in each grid
from its own loads**. Using the real grid's 20 IDs in a random grid would not be fair: there they are ordinary nodes,
not hubs, so the datacenter's own connection point overloads at once.

**Measured per location:** demand threshold (smallest demand that overloads a substation) and failures / LCC at demand 60.

**Needs:** `data/processed/nordic_graph.pkl`, `data/processed/nulls/`, `src/cascade.py`.
**Writes:** `results/null_cascade.csv`, `results/null_cascade_summary.csv`, `results/null_cascade_zscores.csv`,
`results/figures/null_cascade_placement.png`.

Runtime: about 100 s per random grid; runs in parallel on all cores (≈ 10–25 min on a laptop). Set `N_NULLS` lower to test.

In [ ]:
import sys, pickle, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

REPO_ROOT = Path.cwd().parent
sys.path.append(str(REPO_ROOT))
from src.cascade import to_igraph, loads, add_datacenter, cascade, demand_threshold

PROCESSED = REPO_ROOT / "data" / "processed"
NULLS     = PROCESSED / "nulls"
RESULTS   = REPO_ROOT / "results"
FIGURES   = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

ALPHA, DEMAND = 0.2, 60
N_LOCATIONS = 20       # same as Sergiu's placeholder list
N_NULLS = 30           # set to 2-3 for a quick test
print("REPO_ROOT:", REPO_ROOT)

## One grid: pick the 20 locations, then threshold + cascade at each

`null_models.ipynb` saved `cost = 1/c`; Sergiu's code expects the integer cost from `build_graph.py`, so it is recomputed.
The imports sit inside the function so the parallel workers can find `src/`.

In [ ]:
def run_grid(path, alpha=ALPHA, demand=DEMAND, n_loc=N_LOCATIONS, repo=str(REPO_ROOT)):
    import sys, pickle, time
    sys.path.append(repo)
    from src.cascade import to_igraph, loads, add_datacenter, cascade, demand_threshold

    t = time.time()
    G = pickle.load(open(path, "rb"))
    for _, _, d in G.edges(data=True):
        d["cost"] = int(round(1_000_000 / d["capacity"]))
    g = to_igraph(G)
    L0 = loads(g)

    hv = [n for n, d in G.nodes(data=True) if d.get("voltage", 0) >= 380]
    locations = sorted(hv, key=L0.get, reverse=True)[:n_loc]

    rows = []
    for rank, bus in enumerate(locations, start=1):
        th, first = demand_threshold(g, bus, alpha, L0)
        failed, lcc = cascade(g, alpha, h=add_datacenter(g, bus), dc="DC", demand=demand, L0=L0)
        rows.append({"location": bus, "load_rank": rank, "country": G.nodes[bus]["country"],
                     "degree": G.degree(bus), "load_L0": L0[bus],
                     "threshold": th, "first_to_fail": first,
                     "failed": len(failed), "lcc": lcc})
    return rows, float(np.mean(list(L0.values()))), time.time() - t

## Run: real grid + random grids, in parallel

In [ ]:
paths = [PROCESSED / "nordic_graph.pkl"] + sorted(NULLS.glob("null_*.pkl"))[:N_NULLS]
names = ["real"] + [p.stem for p in paths[1:]]

t0 = time.time()
out = Parallel(n_jobs=-1, verbose=5)(delayed(run_grid)(str(p)) for p in paths)
print(f"done in {(time.time() - t0) / 60:.1f} min")

rows, mean_load = [], {}
for name, (r, ml, _) in zip(names, out):
    mean_load[name] = ml
    rows += [{"graph": name, **x} for x in r]
df = pd.DataFrame(rows)
df.to_csv(RESULTS / "null_cascade.csv", index=False)

real = df[df.graph == "real"]
real[["location", "country", "threshold", "failed", "lcc"]].round(3)

**Check:** the real-grid rows must match Sergiu's `datacenter_immediate.csv` (demand 60) and his thresholds (28–100).

In [ ]:
sergiu = pd.read_csv(RESULTS / "datacenter_immediate.csv")
sergiu = sergiu[sergiu.demand == DEMAND].set_index("candidate")["failed"]
both = real.set_index("location")["failed"].align(sergiu, join="inner")
print(f"matching locations: {len(both[0])} / {len(real)},  identical failures: {(both[0] == both[1]).all()}")
print(f"real thresholds: {real.threshold.min():.0f}–{real.threshold.max():.0f}  (Sergiu: 28–100)")

## Placement effect per grid

- **threshold min / median / max**: in demand units (absolute)
- **threshold CV** (std / mean) and **max / min**: how much the location matters, *independent of scale*
- **collapses at demand 60**: locations where more than half of the grid is cut off (LCC < 50 %)
- **no failures at demand 60**: locations where the grid absorbs the datacenter

Note: loads (betweenness) scale with path length, and random grids have ~2.6× shorter paths, so their loads and
therefore capacities are lower (mean load below). Absolute thresholds are not directly comparable; CV and max/min are.

In [ ]:
def summarise(d):
    th = d.threshold
    return pd.Series({
        "mean_load_L0": mean_load[d.name],
        "threshold_min": th.min(), "threshold_median": th.median(), "threshold_max": th.max(),
        "threshold_cv": th.std() / th.mean(), "threshold_max_over_min": th.max() / th.min(),
        "collapses_at_demand": int((d.lcc < 0.5).sum()),
        "no_failures_at_demand": int((d.failed == 0).sum()),
        "failed_mean": d.failed.mean(),
    })

summ = df.groupby("graph", sort=False).apply(summarise, include_groups=False)
summ.to_csv(RESULTS / "null_cascade_summary.csv")
summ.round(2)

## Real vs. random grids: z-scores

In [ ]:
r, n = summ.loc["real"], summ.drop(index="real")
z = pd.DataFrame({"real": r, "null_mean": n.mean(), "null_std": n.std(ddof=1)})
z["z"] = (z.real - z.null_mean) / z.null_std.replace(0, np.nan)
z["p_empirical"] = [(np.abs(n[m] - n[m].mean()) >= abs(r[m] - n[m].mean())).mean() for m in z.index]
z.to_csv(RESULTS / "null_cascade_zscores.csv")
z.round(3)

## Figure

Left: absolute thresholds per location (each row = one grid). Middle: thresholds divided by that grid's median
(the scale-free placement effect). Right: failures at demand 60.

In [ ]:
grids = names
y = {g: i for i, g in enumerate(grids)}
fig, axes = plt.subplots(1, 3, figsize=(13, 6.5), sharey=True)
df["threshold_rel"] = df.threshold / df.groupby("graph").threshold.transform("median")

for ax, col, label in [(axes[0], "threshold", "Demand threshold"),
                       (axes[1], "threshold_rel", "Threshold / median of that grid"),
                       (axes[2], "failed", f"Failed substations at demand {DEMAND}")]:
    for g_, d in df.groupby("graph", sort=False):
        is_real = g_ == "real"
        ax.scatter(d[col], np.full(len(d), y[g_]), s=30 if is_real else 12,
                   color="#c8553d" if is_real else "#2a78b5", alpha=0.9 if is_real else 0.45,
                   marker="D" if is_real else "o", lw=0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xlabel(label)
axes[0].axvline(DEMAND, color="grey", ls=":", lw=1)
axes[0].text(DEMAND, 1.0, f" demand {DEMAND}", fontsize=8, color="grey",
             transform=axes[0].get_xaxis_transform(), va="bottom")
axes[0].set_yticks(range(len(grids)), grids, fontsize=7)
axes[0].invert_yaxis()
fig.suptitle(f"Datacenter placement: real grid (red) vs. random grids (blue), α = {ALPHA}, "
             f"{N_LOCATIONS} locations per grid", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "null_cascade_placement.png", dpi=200)
plt.show()

## Line for the Results log

In [ ]:
print(f"Null-model cascade (α={ALPHA}, demand {DEMAND}, top-{N_LOCATIONS} ≥380 kV hubs per grid, {len(n)} random grids): "
      f"thresholds real {r.threshold_min:.0f}–{r.threshold_max:.0f} vs random {n.threshold_min.mean():.0f}–{n.threshold_max.mean():.0f} "
      f"(loads {r.mean_load_L0 / n.mean_load_L0.mean():.1f}× higher in the real grid); "
      f"spread across locations CV {r.threshold_cv:.2f} vs {n.threshold_cv.mean():.2f}. "
      f"At demand {DEMAND}: real {int(r.collapses_at_demand)}/{N_LOCATIONS} locations collapse and "
      f"{int(r.no_failures_at_demand)} have no failures; random grids {n.collapses_at_demand.mean():.1f}/{N_LOCATIONS} collapse, "
      f"{n.no_failures_at_demand.mean():.1f} without failures.")